# Answer quality review

Run the cells in order to review the saved answers against [25 question-specific checklists](../evaluation/answer_checklists.json). Each requirement receives a focused yes/no/uncertain assessment with a short explanation. Requirements for each question are batched into one model request to avoid repeated overhead.

The report leads with **what went well, what needs attention, and the next step**. It shows at most two brief source excerpts copied by the code, not written by the reviewer. Detailed answers and automatic checks are collapsed below each summary; complete data stays in JSON.

Use the project's Python kernel, start Ollama, and choose an installed model. Set `CHECKS_ONLY = True` to run citation, number, length, and retrieval checks without an LLM (use a separate output filename). Set `LIMIT = 1` for a pilot; `0` reviews all saved answers. Content review can still take time locally.

The checklists are assistant-prepared from the local papers and **pending human approval**. This focused reviewer is advisory and needs calibration; it does not check every possible factual error. See the [evaluation guide](../evaluation/README.md).

The initial focused pilot caught important omissions but still overcredited some requirements. Read the [calibration notes](../evaluation/results/checklist_calibration.md); a “Covered” label is not final approval.


## Selected AI questions

This run reviews three fresh `llama3.2` answers from different AI papers, using their existing frozen retrieval passages and `qwen3.5:9b` as reviewer:

| ID | Question | What it tests |
| --- | --- | --- |
| q08 | How do submissions that change the learning algorithm compare with those that do not? | Numerical comparison in AI4AI-Bench |
| q09 | What two kinds of task structure does TMI recover from computer-use traces? | Explaining task-model induction |
| q18 | How does G-CARL supervise factuality and response coverage? | Evidence verification and checklist coverage |

The earlier Router/Bidder results remain in their original files. The new report is saved to `evaluation/results/checklist_ai_three.md`.

To reproduce answer generation from the repository root (completed answers resume):

```bash
.venv/bin/python evaluation/run_answers.py --retrieval evaluation/results/retrieval_ordered_frozen.json --ids q08 q09 q18 --model llama3.2 --output evaluation/results/answers_ai_three.json
```


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import urllib.request
from IPython.display import Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "evaluation/review_answers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the repository.")
print(f"Repository: {ROOT}")
print(f"Python: {sys.executable}")


Repository: /Users/alexnguyen/Documents/GitHub/KPMG-1C-ai-research-intelligence-agent-for-business-insight-translation
Python: /opt/anaconda3/bin/python


In [2]:
# Edit these settings before running the judge.
ANSWERS = ROOT / "evaluation/results/answers_ai_three.json"
MODEL = "qwen3.5:9b"
HOST = "http://127.0.0.1:11434"
LIMIT = 0                 # 0 = all saved answers (currently 3); otherwise first N
RETRIES = 0               # 1 enables one repair attempt after invalid output
NUM_CTX = 16384
NUM_PREDICT = 1400         # Output ceiling; lowering it may truncate the JSON
TIMEOUT = 600              # Seconds per model request
OUTPUT = ROOT / "evaluation/results/checklist_ai_three.json"

# Use a new OUTPUT filename when changing model, inputs, options, or script code.
# Compatible runs reuse successful reviews; failed reviews are attempted again.

CHECKS_ONLY = False        # True = fast checks, no model request
CHECKLISTS = ROOT / "evaluation/answer_checklists.json"
MAX_WORDS = 120            # Business readability target


## Check inputs and model availability

Shows how many answers will be evaluated. In checks-only mode this skips Ollama. No model is downloaded.


In [3]:
if not ANSWERS.is_file():
    raise FileNotFoundError(f"Saved answers not found: {ANSWERS}")
answer_data = json.loads(ANSWERS.read_text())
print(f"Saved answers: {len(answer_data['answers'])}")
selected = answer_data["answers"][:LIMIT or None]
print(f"Selected for evaluation: {len(selected)} ({', '.join(a['id'] for a in selected)})")
if not CHECKS_ONLY:
    try:
        with urllib.request.urlopen(HOST.rstrip('/') + '/api/tags', timeout=10) as response:
            models = json.load(response)['models']
    except OSError as exc:
        raise RuntimeError("Cannot reach Ollama. Start Ollama and check HOST.") from exc
    installed = [model['name'] for model in models]
    print("Installed models:", ", ".join(installed))
    if MODEL not in installed and MODEL + ':latest' not in installed:
        raise ValueError(f"Choose an installed MODEL. {MODEL!r} is not available.")


Saved answers: 3
Selected for evaluation: 3 (q08, q09, q18)
Installed models: llama3.2:latest, gpt-oss:20b, llama3.1:8b, qwen3.5:9b, qwen2.5:latest, llama3:latest, nomic-embed-text:latest, hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest, hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest, llama2:latest


## Run the review

Progress streams below. Compatible runs reuse completed reviews. Use a new output filename after changing model, checklist, script, or settings. Interrupting stops the child process; completed answers are checkpointed.


In [4]:
command = [
    sys.executable, '-u', str(ROOT / 'evaluation/review_answers.py'),
    '--answers', str(ANSWERS), '--model', MODEL, '--host', HOST,
    '--checklists', str(CHECKLISTS), '--max-words', str(MAX_WORDS),
    '--limit', str(LIMIT), '--retries', str(RETRIES),
    '--num-ctx', str(NUM_CTX), '--num-predict', str(NUM_PREDICT),
    '--timeout', str(TIMEOUT), '--output', str(OUTPUT),
]
if CHECKS_ONLY:
    command.append('--checks-only')
# Stream output rather than hiding progress until the subprocess completes.
run_finished = False
run_log = []  # Full CLI text, available for troubleshooting
process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        run_log.append(line)
        if line.startswith(('Reviewing ', '  Focused', '  Completed', '  Review unavailable', 'Error:', 'Report:', 'Details:')):
            print(line, end='', flush=True)
    return_code = process.wait()
    run_finished = True
except BaseException:
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
finally:
    process.stdout.close()

if return_code:
    print("The run reported errors. Check the messages above; any completed reviews are saved.")


Reviewing q08 (2 required points)...


  Focused review attempt 1/1...


  Completed in 20.2s.


Reviewing q09 (2 required points)...


  Focused review attempt 1/1...


  Completed in 12.8s.


Reviewing q18 (2 required points)...


  Focused review attempt 1/1...


  Completed in 13.1s.


Report: /Users/alexnguyen/Documents/GitHub/KPMG-1C-ai-research-intelligence-agent-for-business-insight-translation/evaluation/results/checklist_ai_three.md


Details: /Users/alexnguyen/Documents/GitHub/KPMG-1C-ai-research-intelligence-agent-for-business-insight-translation/evaluation/results/checklist_ai_three.json


## Read the formatted report

Run setup and configuration, then this cell to open an existing report **without rerunning the model**. Expand “Answer and automatic checks” only if more detail is needed. A review failure is explained separately from an answer defect.


In [5]:
if not OUTPUT.is_file():
    raise RuntimeError("No report was written. Resolve the error above and rerun.")
if globals().get('return_code', 0):
    display(Markdown("**The judge run reported errors. The saved checkpoint below may include earlier results; check the run log above.**"))

report = json.loads(OUTPUT.read_text())
# Use the script's renderer so the notebook and CLI show the same assessments.
import importlib.util
spec = importlib.util.spec_from_file_location('notebook_judge', ROOT / 'evaluation/review_answers.py')
judge = importlib.util.module_from_spec(spec)
spec.loader.exec_module(judge)
if report.get('schema_version') != 2:
    raise ValueError('This is an older broad-judge report. Select the new checklist report path.')
display(Markdown(judge.render_report(report)))
print(f"JSON checkpoint: {OUTPUT}")
print(f"Markdown report: {OUTPUT.with_suffix('.md')}")


# Answer quality review

3 answers · 3 content reviews completed.

Checklist-based automated advice, not final approval. Checklists are assistant-prepared and pending human approval. The reviewer checks required points, not every possible factual error.

## q08 — Needs attention

How do submissions that change the learning algorithm compare with those that do not?

**What went well:** Algorithm-changing submissions covered, according to the reviewer.

**What needs attention:** Other submissions: Answer fails to report the specific score of 0.126 for other submissions as required. Check figures against their sources.

**Next step:** Correct the items below, then review again.

**Sources found:** 1 of 1 expected papers. Finding the right paper does not prove the answer is correct.

| Required point | Result | Short explanation |
| --- | --- | --- |
| Algorithm-changing submissions | Covered | Answer reports mean normalized score of 0.226 for algorithm-changing submissions. |
| Other submissions | Needs fixing | Answer fails to report the specific score of 0.126 for other submissions as required. |

**Source excerpts to check** — copied from saved passages; shortened to 240 characters. The reviewer selected these locations; relevance still needs checking.

**Source 2 — Other submissions**

> …and capacity instead. The 122 that do reach it — the objective, the supervision signal, the learning rule, the data — average0.226 against0 .126for the rest: the algorithmic layer is where the distance gets closed, and most submissions…


<details><summary>Answer and automatic checks</summary>

Submissions that change the learning algorithm (122 submissions) do not differ significantly from those that do not (141 submissions) in terms of the number of submissions that reach the learning procedure, with 46.4% of the former and 53.6% of the latter reaching the procedure. [3]  The provided passages do not provide a direct comparison between the two groups.

- Source references exist; whether they support each claim still needs review.
- Check figures against their sources.
- Answer is 57 words; the review target is at most 120.

Checks do not establish factual accuracy. Full source passages, ratings, and diagnostics are in the JSON file.

</details>

## q09 — Checklist met — spot-check recommended

What two kinds of task structure does TMI recover from computer-use traces?

**What went well:** Goals; Procedure covered, according to the reviewer.

**What needs attention:** No checklist issues identified.

**Next step:** Have a person spot-check the answer and its sources.

**Sources found:** 1 of 1 expected papers. Finding the right paper does not prove the answer is correct.

| Required point | Result | Short explanation |
| --- | --- | --- |
| Goals | Covered | Answer explicitly identifies the objective model as a hierarchical decomposition of objectives. |
| Procedure | Covered | Answer explicitly identifies the procedure model as specifying execution control flow via sequencing and iteration. |

**Source excerpts to check** — copied from saved passages; shortened to 240 characters. The reviewer selected these locations; relevance still needs checking.

**Source 3 — Goals**

> for each task t∈ T, inducing a task model Mt that unifies two complementary axes. • Theobjective model Ot is a hierarchical decomposition of the objective of t into the subobjectives the user pursues. • Theprocedure model Pt is a…

**Source 4 — Goals**

> …which (i) discovers the latent tasks in an unconstrained trace, disentangling concurrent activity, and (ii) for each latent task, induces a task model pairing a hierarchical objective model of recursive goal decomposition with a…


<details><summary>Answer and automatic checks</summary>

TMI recovers two kinds of task structure: objective models and procedure models. [3][2]  The objective model is a hierarchical decomposition of the objective of a task into subobjectives the user pursues. [3]  The procedure model is a trace-grounded composition of control-flow operators specifying how the execution of a task is organized through sequencing and iteration. [3]

- Source references exist; whether they support each claim still needs review.
- Answer is 53 words; the review target is at most 120.

Checks do not establish factual accuracy. Full source passages, ratings, and diagnostics are in the JSON file.

</details>

## q18 — Needs attention

How does G-CARL supervise factuality and response coverage?

**What went well:** Factual support; Coverage covered, according to the reviewer.

**What needs attention:** Add source references to the answer itself. Check figures against their sources.

**Next step:** Correct the items below, then review again.

**Sources found:** 1 of 1 expected papers. Finding the right paper does not prove the answer is correct.

| Required point | Result | Short explanation |
| --- | --- | --- |
| Factual support | Covered | Answer states verification uses evidence from a multi-source medical datastore. |
| Coverage | Covered | Answer describes constructing case-specific weighted checklists via MLLM and clinician refinement. |

**Source excerpts to check** — copied from saved passages; shortened to 240 characters. The reviewer selected these locations; relevance still needs checking.

**Source 4 — Factual support**

> …a user queryq,thepolicymodelfirstgeneratescandidateresponses. The retrieval-grounded branch then supervises medical factuality by verifying report-grounded medical claims against evidence retrieved from a multi-source medical datastore,…

**Source 3 — Factual support**

> …conventional supervised fine-tuning and holistic reinforcement learning paradigms. To address this challenge, we propose G-CARL, a grounded, checklist-aligned reinforcementlearningframeworkthatcombinesmulti-source retrieval for atomic…


<details><summary>Answer and automatic checks</summary>

G-CARL supervises factuality through a retrieval-grounded branch that verifies report-grounded medical claims against evidence retrieved from a multi-source medical datastore, producing the factuality reward Rfact (Sec. 3.2).   It also constructs a case-specific weighted checklist through MLLM generation followed by clinician-guided refinement, and evaluates the checklist coverage of the generated response to produce the checklist reward Rcheck (Sec. 3.3).

- Add source references to the answer itself.
- Check figures against their sources.
- Answer is 58 words; the review target is at most 120.

Checks do not establish factual accuracy. Full source passages, ratings, and diagnostics are in the JSON file.

</details>


JSON checkpoint: /Users/alexnguyen/Documents/GitHub/KPMG-1C-ai-research-intelligence-agent-for-business-insight-translation/evaluation/results/checklist_ai_three.json
Markdown report: /Users/alexnguyen/Documents/GitHub/KPMG-1C-ai-research-intelligence-agent-for-business-insight-translation/evaluation/results/checklist_ai_three.md


## Spot-check of this run

These are **assistant observations against the saved answers and passages**, separate from the model's ratings; they are not an independent human review.

| Answer | Practical takeaway |
| --- | --- |
| **q08 · AI4AI-Bench** | **Rewrite the comparison.** The answer confuses submission counts/proportions with performance. Source 2 reports mean scores of **0.226 versus 0.126**. Both are missing from the answer. The judge incorrectly marked the first value “Covered”; do not accept that rating. |
| **q09 · Task Model Induction** | **The two required structures are covered:** a hierarchical objective model and a procedure model describing execution control flow. |
| **q18 · G-CARL** | **Both mechanisms are described, but source citations are missing.** The automatic number warnings include “3.2” and “3.3”, which are section references, not necessarily incorrect research figures. |

All three model reviews completed without processing errors in about **46 seconds of model time**. Completion does not guarantee correctness, as q08 demonstrates. Original model ratings remain unchanged in the report and JSON for auditability.
